In [ ]:
# Import the necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import sys


In [2]:
# Find the project root
PROJECT_ROOT = Path.cwd().resolve().parents[1]

# Make the project root available to Python
sys.path.insert(0, str(PROJECT_ROOT))

# Load dataset
df = pd.read_csv(r"..\..\data\customers.csv") 

In [ ]:

from src.data_cleaning import inspect_dataset

inspect_dataset(df)


1. FIRST 3 ROWS
  customer_id first_name last_name gender age_band       phone home_store_id  \
0      C00001     Winnie    Mwangi      F    25-34  0787118070          S016   
1      C00002       Eric    Mwende      M    45-54  0726805038          S016   
2      C00003        Ann     Kamau      F    18-24  0785524002          S025   
3      C00004      James    Omondi      M    25-34  0784212481          S024   
4      C00005      Mercy      Juma      F    35-44  0700993331          S014   

    county signup_date   segment_true  
0   Kisumu  2022-04-17  Budget Family  
1   Kisumu  2023-09-11  Budget Family  
2     Meru  2026-07-30     Occasional  
3     Meru  2022-08-10  Budget Family  
4  Mombasa  2024-03-26  Budget Family  

2. DATASET SHAPE
Number of rows: 6000
Number of columns: 10

3. COLUMN NAMES
['customer_id', 'first_name', 'last_name', 'gender', 'age_band', 'phone', 'home_store_id', 'county', 'signup_date', 'segment_true']

4. DUPLICATE ROWS
Number of duplicate rows: 0

6. M

In [6]:
# 1. HANDLE MISSING AGE BANDS

missing_ages_bf = df["age_band"].isnull().sum()

# Fill in the missing bands => unknown
df["age_band"] = df["age_band"].fillna("unknown")

missing_ages_af = df["age_band"].isnull().sum()


print("Missing ages before:", missing_ages_bf)
print("Missing ages after:", missing_ages_af)

Missing ages before: 180
Missing ages after: 0


In [8]:
# 2. HANDLE INCONSISTENT COUNTIES

counties = df.groupby("county")["county"].count().sort_values(ascending=False)
print(counties)



county
Nairobi         1559
Kiambu          1038
Mombasa          619
Nakuru           507
Kakamega         484
Kilifi           274
Nyeri            266
Meru             246
Kajiado          236
Machakos         207
Uasin Gishu      137
Kisumu           127
NAIROBI CITY      89
KIAMBU            54
MOMBASA           40
NAKURU            28
KAKAMEGA          26
NYERI             16
MERU              13
KAJIADO            9
KILIFI             9
MACHAKOS           8
KISUMU             4
UASIN-GISHU        4
Name: county, dtype: int64


In [9]:
# Replace values

values = {
    "NAIROBI CITY": "Nairobi",
    "KIAMBU": "Kiambu",
    "MOMBASA": "Mombasa",
    "NAKURU": "Nakuru",
    "KAKAMEGA": "Kakamega",
    "NYERI": "Nyeri",
    "MERU": "Meru",
    "KAJIADO": "Kajiado",
    "KILIFI": "Kilifi",
    "MACHAKOS": "Machakos",
    "KISUMU": "Kisumu",
    "UASIN-GISHU": "Uasin Gishu"
}

df["county"] = df["county"].str.strip().replace(values, regex=False)

counties = df.groupby("county")["county"].count().sort_values(ascending=False)
print(counties)

county
Nairobi        1648
Kiambu         1092
Mombasa         659
Nakuru          535
Kakamega        510
Kilifi          283
Nyeri           282
Meru            259
Kajiado         245
Machakos        215
Uasin Gishu     141
Kisumu          131
Name: county, dtype: int64


In [10]:
# 2. HANDLE PHONE NUMBER FORMAT

# Remove country code and start with 07..

df["phone"] = df["phone"].astype(str).str.replace(r'^(\+?254)', '0', regex=True)

print(df.head(10))

  customer_id first_name  last_name gender age_band       phone home_store_id  \
0      C00001     Winnie     Mwangi      F    25-34   787118070          S016   
1      C00002       Eric     Mwende      M    45-54  0726805038          S016   
2      C00003        Ann      Kamau      F    18-24   785524002          S025   
3      C00004      James     Omondi      M    25-34   784212481          S024   
4      C00005      Mercy       Juma      F    35-44   700993331          S014   
5      C00006     Winnie   Odhiambo      F    45-54  0769322217          S003   
6      C00007      Diana  Kiprotich      F    18-24   735941197          S008   
7      C00008     Martin   Odhiambo      M    25-34   751617308          S009   
8      C00009   Beatrice   Odhiambo      F    18-24  0797748149          S001   
9      C00010       Ruth      Kirui      F    35-44   756510011          S008   

    county signup_date        segment_true  
0   Kisumu  2022-04-17       Budget Family  
1   Kisumu  2023-0

In [12]:
# Add a leading 0 to numbers without

df["phone"] = np.where(
    df["phone"].str.startswith('0'),
    df["phone"],
    '0' + df["phone"]
)

print(df.head(10))

  customer_id first_name  last_name gender age_band       phone home_store_id  \
0      C00001     Winnie     Mwangi      F    25-34  0787118070          S016   
1      C00002       Eric     Mwende      M    45-54  0726805038          S016   
2      C00003        Ann      Kamau      F    18-24  0785524002          S025   
3      C00004      James     Omondi      M    25-34  0784212481          S024   
4      C00005      Mercy       Juma      F    35-44  0700993331          S014   
5      C00006     Winnie   Odhiambo      F    45-54  0769322217          S003   
6      C00007      Diana  Kiprotich      F    18-24  0735941197          S008   
7      C00008     Martin   Odhiambo      M    25-34  0751617308          S009   
8      C00009   Beatrice   Odhiambo      F    18-24  0797748149          S001   
9      C00010       Ruth      Kirui      F    35-44  0756510011          S008   

    county signup_date        segment_true  
0   Kisumu  2022-04-17       Budget Family  
1   Kisumu  2023-0

In [18]:
# CHANGE THE DATE DATATYPE 

df["signup_date"] = pd.to_datetime(df["signup_date"])

print(df["signup_date"].dtype)


datetime64[us]


In [20]:
# SAVE AND EXPORT THE CLEANED DATASET

output_dir = Path('../../data/processed')
output_dir.mkdir(parents=True, exist_ok=True)

df.to_csv(
    output_dir / 'customers_cleaned.csv',
    index=False
)